In [ ]:
import numpy as np
import pandas as pd

def calculate_maut(metrics):
    """
    Takes a dictionary of raw performance metrics and returns
    the individual utilities and the final MAUT score.
    """
    # 1. Define Bounds (Best and Worst values from our space)
    bounds = {
        'P1': {'best': 12.00, 'worst': 13.71},
        'P2': {'best': 807.5, 'worst': 373.3},
        'P3': {'best': 0.811, 'worst': 0.993},
        'P4': {'best': 100.00, 'worst': 98.00}
    }

    # Extract raw values
    x_p1 = metrics['P1_Efficiency_MJ_per_mile']
    x_p2 = metrics['P2_Range_miles']
    x_p3 = metrics['P3_CO2_per_mile']
    x_p4 = metrics['P4_Reliability_pct']

    # 2. Calculate Utilities
    # U_P1: Linear (Lower is Better)
    u_p1 = (bounds['P1']['worst'] - x_p1) / (bounds['P1']['worst'] - bounds['P1']['best'])

    # U_P2: Linear (Higher is Better)
    u_p2 = (x_p2 - bounds['P2']['worst']) / (bounds['P2']['best'] - bounds['P2']['worst'])

    # U_P3: Logarithmic (Lower is Better)
    ratio_p3 = (x_p3 - bounds['P3']['best']) / (bounds['P3']['worst'] - bounds['P3']['best'])
    u_p3 = 1.0 - (np.log(1.0 + ratio_p3) / np.log(2.0))

    # U_P4: Linear (Higher is Better)
    u_p4 = (x_p4 - bounds['P4']['worst']) / (bounds['P4']['best'] - bounds['P4']['worst'])

    # Ensure utilities are clamped between 0 and 1 (in case of slight rounding errors)
    u_p1, u_p2, u_p3, u_p4 = [max(0.0, min(1.0, u)) for u in (u_p1, u_p2, u_p3, u_p4)]

    # 3. Calculate Overall Utility (Weighted Sum)
    w1, w2, w3, w4 = 0.30, 0.15, 0.25, 0.30
    overall_utility = (w1 * u_p1) + (w2 * u_p2) + (w3 * u_p3) + (w4 * u_p4)

    return {
        'U_Efficiency': round(u_p1, 3),
        'U_Range': round(u_p2, 3),
        'U_Emissions': round(u_p3, 3),
        'U_Reliability': round(u_p4, 3),
        'Overall_Utility': round(overall_utility, 3)
    }

# Raw metrics generated from Question 3
raw_data = {
    'Ref Arch 1': {'P1_Efficiency_MJ_per_mile': 12.00, 'P2_Range_miles': 382.3, 'P3_CO2_per_mile': 0.811, 'P4_Reliability_pct': 98.00},
    'Ref Arch 2': {'P1_Efficiency_MJ_per_mile': 12.00, 'P2_Range_miles': 373.3, 'P3_CO2_per_mile': 0.829, 'P4_Reliability_pct': 99.96},
    'Ref Arch 3': {'P1_Efficiency_MJ_per_mile': 13.71, 'P2_Range_miles': 807.5, 'P3_CO2_per_mile': 0.993, 'P4_Reliability_pct': 100.00} # Clamped to 100 max
}

# Run MAUT Evaluation
maut_results = []
for arch_name, metrics in raw_data.items():
    res = calculate_maut(metrics)
    res['Architecture'] = arch_name
    maut_results.append(res)

df_maut = pd.DataFrame(maut_results).set_index('Architecture')
print(df_maut)

              U_Efficiency  U_Range  U_Emissions  U_Reliability  \
Architecture                                                      
Ref Arch 1             1.0    0.021        1.000           0.00   
Ref Arch 2             1.0    0.000        0.864           0.98   
Ref Arch 3             0.0    1.000        0.000           1.00   

              Overall_Utility  
Architecture                   
Ref Arch 1              0.553  
Ref Arch 2              0.810  
Ref Arch 3              0.450  
